In [ ]:
import pandas as pd
import numpy as np
import os
import joblib
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

ModuleNotFoundError: No module named 'matplotlib'

# Load data

In [ ]:
maison_df = pd.read_csv('datasets_prepd/dvf_maison.csv')
print(maison_df.head())

    id_mutation date_mutation  numero_disposition nature_mutation  \
0  2021-1289503    2021-01-05                   1           Vente   
1  2021-1289509    2021-01-06                   1           Vente   
2  2021-1289511    2021-01-04                   1           Vente   
3  2021-1289514    2021-01-05                   1           Vente   
4  2021-1289517    2021-01-04                   1           Vente   

   valeur_fonciere  adresse_numero adresse_suffixe      adresse_nom_voie  \
0         352000.0           228.0             NaN       RUE DE L EGLISE   
1         334800.0             7.0             NaN          AV ROSCOMMON   
2         225700.0            35.0             NaN  RUE DE LA REPUBLIQUE   
3         224000.0             7.0             NaN     CHE DE LA GARENNE   
4         267000.0            11.0             NaN     ALL DES AUBEPINES   

  adresse_code_voie  code_postal  ...  nature_culture_speciale  \
0              0220      77115.0  ...                      NaN

# Features

In [ ]:
features = [
    "longitude",
    "latitude",
    "code_postal",
    "surface_reelle_bati",
    "nombre_pieces_principales",
    "surface_terrain",
    "number_of_lots",
    "season"
]

target = "valeur_fonciere_actualisee"

# Encode season to numeric values
season_mapping = {"winter": 0, "spring": 1, "summer": 2, "autumn": 3}
maison_df["season"] = maison_df["season"].map(season_mapping)

X = maison_df[features]
y = maison_df[target]

print("Number of features:", X.shape[1])
print(X.head())
print(y.head())

# Make sets : trail and test and validate

In [ ]:
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y,
    test_size=0.3,
    random_state=42
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp,
    test_size=0.5,
    random_state=42
)

print("Train:", X_train.shape)
print("Validation:", X_val.shape)
print("Test:", X_test.shape)

Train: (26978, 9)
Validation: (5781, 9)
Test: (5782, 9)


# Ssave

In [ ]:
os.makedirs("data_maison", exist_ok=True)

train_df = X_train.copy()
val_df = X_val.copy()
test_df = X_test.copy()

train_df[target] = y_train
val_df[target] = y_val
test_df[target] = y_test

train_df.to_csv("data_maison/maison_train.csv", index=False)
val_df.to_csv("data_maison/maison_val.csv", index=False)
test_df.to_csv("data_maison/maison_test.csv", index=False)

print("Datasets saved.")

Datasets saved.


Train model

In [ ]:
# Baseline CV score
baseline_rf = RandomForestRegressor(n_estimators=200, max_depth=20, random_state=42, n_jobs=-1)
cv_scores = cross_val_score(baseline_rf, X_train, y_train, cv=5, scoring='r2')
print(f"Baseline CV R2: {cv_scores.mean():.4f} (+/- {cv_scores.std():.4f})")

# GridSearchCV for regularized hyperparameters
rf = RandomForestRegressor(random_state=42, n_jobs=-1)
param_grid = {
    'n_estimators': [200, 300],
    'max_depth': [8, 10, 12],
    'min_samples_split': [5, 10, 20],
    'min_samples_leaf': [4, 8],
}

grid_search = GridSearchCV(rf, param_grid, cv=5, scoring='r2', verbose=1, n_jobs=1)
grid_search.fit(X_train, y_train)

model = grid_search.best_estimator_
print(f"\nBest params: {grid_search.best_params_}")
print(f"Best CV R2: {grid_search.best_score_:.4f}")

# Validation

In [ ]:
importances = model.feature_importances_
feat_imp = pd.Series(importances, index=features).sort_values(ascending=True)

plt.figure(figsize=(8, 5))
feat_imp.plot(kind='barh')
plt.title("Feature Importance - Maisons")
plt.xlabel("Importance")
plt.tight_layout()
plt.show()

print("\nFeature importances:")
for name, imp in feat_imp.sort_values(ascending=False).items():
    print(f"  {name}: {imp:.4f}")

# Evaluation

In [ ]:
# Evaluate on all 3 sets
for name, X_set, y_set in [("Train", X_train, y_train), ("Validation", X_val, y_val), ("Test", X_test, y_test)]:
    y_pred = model.predict(X_set)
    rmse = np.sqrt(mean_squared_error(y_set, y_pred))
    mae = mean_absolute_error(y_set, y_pred)
    r2 = r2_score(y_set, y_pred)
    erreur_pct = (abs(y_set - y_pred) / y_set * 100).mean()
    print(f"{name:10s} — RMSE: {rmse:>10.0f} | MAE: {mae:>10.0f} | R2: {r2:.4f} | Erreur moy: {erreur_pct:.2f}%")

# Overfitting gap
r2_train = r2_score(y_train, model.predict(X_train))
r2_val = r2_score(y_val, model.predict(X_val))
gap = r2_train - r2_val
print(f"\nGap surapprentissage (Train R2 - Val R2): {gap:.4f}")

# Evaluation


In [ ]:
gb = HistGradientBoostingRegressor(
    max_iter=300, max_depth=6, learning_rate=0.1,
    min_samples_leaf=10, random_state=42
)
gb.fit(X_train, y_train)

for name, X_set, y_set in [("Train", X_train, y_train), ("Validation", X_val, y_val), ("Test", X_test, y_test)]:
    y_pred = gb.predict(X_set)
    r2 = r2_score(y_set, y_pred)
    erreur_pct = (abs(y_set - y_pred) / y_set * 100).mean()
    print(f"GB {name:10s} — R2: {r2:.4f} | Erreur moy: {erreur_pct:.2f}%")

gb_gap = r2_score(y_train, gb.predict(X_train)) - r2_score(y_val, gb.predict(X_val))
print(f"\nGB Gap surapprentissage: {gb_gap:.4f}")

Save model

In [ ]:
os.makedirs("models", exist_ok=True)

joblib.dump(model, "models/maison_random_forest_model.pkl")

print("Model saved.")
print("Features sauvegardées:", list(model.feature_names_in_))

In [ ]:
y_test_pred = model_maison.predict(X_test)

rmse_test = np.sqrt(mean_squared_error(y_test, y_test_pred))
mae_test = mean_absolute_error(y_test, y_test_pred)
r2_test = r2_score(y_test, y_test_pred)

print("Test RMSE:", rmse_test)
print("Test MAE:", mae_test)
print("Test R2:", r2_test)

In [ ]:
df_test_eval = pd.DataFrame({
    "prix_reel": y_test,
    "prix_predit": y_test_pred
})

df_test_eval["erreur_pct"] = abs(df_test_eval["prix_reel"] - df_test_eval["prix_predit"]) / df_test_eval["prix_reel"] * 100

print("Erreur moyenne test (%) :", df_test_eval["erreur_pct"].mean())